## **Reasons for PPML**

Modeled after this paper: https://arxiv.org/pdf/2606.00438

Two layers of variation:
1. **Between-package variation** Packages that adopt / implement security packages may systematically differ from those that do not (we hypothesize that more popular, better maintained packages, more mature packages are more likely to implement security practices.)
2. **Within-package variation**  

Need to rephrase: "To isolate the causal effect of security practices, we model two layers of variation using a PPML framework with two-way fixed effects. First, we account for between-package variation by including package fixed effects, controlling for time-invariant characteristics such as a package's baseline popularity or complexity. Second, we account for within-package, time-varying variation by including time fixed effects to capture platform-wide macro shocks. This ensures our estimates rely strictly on within-package deviations from their respective means."

PPML handles:
1. zero-inflation of dependent variable 
2. overdispersion of dependent variable
3. Clustered SEs on each package

In [ ]:
library(moments)
library(dplyr)
library(tidyr)
library(lubridate)
library(knitr)
library(kableExtra)


# ---- 1. Load panel --------------------------------------------------------
panel <- read.csv("../data/monthly_panel.csv")
raw_df <- read.csv("../data/sc_control_data_final_with_agg_score_and_vuln_count.csv")


raw_df <- raw_df %>%
    mutate(year_month = floor_date(as_date(published_at), "month"))


Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




In [4]:
summary(panel$Vulnerability_count)
summary(raw_df$vulnerability_count)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   0.00    0.00    9.75   21.70   26.00  755.00 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   0.00    4.00   17.00   36.54   41.00  801.00 

In [5]:
glimpse(raw_df)

Rows: 328,864
Columns: 45
$ github_repo                <chr> "https://github.com/kshetline/json-z", "htt…
$ tag_name                   <chr> "v4.0.1", "v4.1.0", "v4.1.1", "v4.2.0", "v4…
$ tag_commit_sha             <chr> "efca0bfc472bb4e39fc04d4f281f6300bf09163f",…
$ Binary.Artifacts_score     <int> 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10,…
$ Binary.Artifacts_reason    <chr> "no binaries found in the repo", "no binari…
$ CI.Tests_score             <int> -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 10…
$ CI.Tests_reason            <chr> "no pull request found", "0 out of 1 merged…
$ Code.Review_score          <int> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 2…
$ Code.Review_reason         <chr> "Found 0/30 approved changesets -- score no…
$ Dangerous.Workflow_score   <int> -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 10,…
$ Dangerous.Workflow_reason  <chr> "no workflows found", "no workflows found",…
$ License_score              <int> 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 10, 10, 10, 1…
$ License_reas

In [6]:

metrics <- c(
  "Aggregate_score", "Binary_Artifacts_score", "Code_Review_score",
  "Contrib_score", "Dangerous_Workflow_score", "DependUpTool_score",
  "Fuzzing_score", "License_score",
  "Maintained_score", "Pinned_Dependencies_score",
  "Security_Policy_score",
  "Token_Permissions_score"
)

raw_metrics <- c(
  "Aggregate_score", "Binary.Artifacts_score", "Code.Review_score", 
  "Contrib_score","Dangerous.Workflow_score", "DependUpTool_score", 
  "Fuzzing_score", "License_score",
  "Maintained_score", "Pinned.Dependencies_score",
  "Security.Policy_score",
  "Token.Permissions_score"
)

colnames(panel)
colnames(raw_df)

[1] "package_name"              "year_month"               
 [3] "Vulnerability_count"       "Release_count"            
 [5] "Aggregate_score"           "Binary_Artifacts_score"   
 [7] "Code_Review_score"         "Contrib_score"            
 [9] "Dangerous_Workflow_score"  "DependUpTool_score"       
[11] "Fuzzing_score"             "License_score"            
[13] "Maintained_score"          "Pinned_Dependencies_score"
[15] "Security_Policy_score"     "Token_Permissions_score"  
[17] "loc"                       "downloads_7_day_total"    
[19] "dependents"                "version_age_days"

[1] "github_repo"                "tag_name"                  
 [3] "tag_commit_sha"             "Binary.Artifacts_score"    
 [5] "Binary.Artifacts_reason"    "CI.Tests_score"            
 [7] "CI.Tests_reason"            "Code.Review_score"         
 [9] "Code.Review_reason"         "Dangerous.Workflow_score"  
[11] "Dangerous.Workflow_reason"  "License_score"             
[13] "License_reason"             "Pinned.Dependencies_score" 
[15] "Pinned.Dependencies_reason" "Security.Policy_score"     
[17] "Security.Policy_reason"     "Token.Permissions_score"   
[19] "Token.Permissions_reason"   "Vulnerabilities_score"     
[21] "Vulnerabilities_reason"     "Contrib_score"             
[23] "Contrib_reason"             "Contrib_details"           
[25] "Maintained_score"           "Maintained_reason"         
[27] "Maintained_details"         "DependUpTool_score"        
[29] "DependUpTool_reason"        "DependUpTool_details"      
[31] "Fuzzing_score"              "githubrepo"                
[33] "published_at"               "project_name"              
[35] "package_name"               "System"                    
[37] "first_release_date"         "tag_name_clean"            
[39] "loc"                        "dependents"                
[41] "downloads_7_day_total"      "version_age_days"          
[43] "Aggregate_score"            "vulnerability_count"       
[45] "year_month"

In [7]:
# ---- 2. Within-variation summary per metric -------------------------------

within_var_summary <- lapply(raw_metrics, function(m) {
  # 1. Calculate row-level NA percentage across the entire raw_df first
  is_missing_row <- is.na(raw_df[[m]]) | raw_df[[m]] == -1
  pct_missing_observations <- round(mean(is_missing_row, na.rm = TRUE) * 100, 1)
  
  raw_df %>%
    filter(!is.na(.data[[m]])) %>%
    group_by(package_name) %>%
    summarise(
      n_obs = n(),
      n_distinct = n_distinct(.data[[m]]),
      # within_sd = sd(.data[[m]]),
      all_zero = all(.data[[m]] == 0),
      all_ten = all(.data[[m]] == 10),
      .groups = "drop"
    ) %>%
    summarise(
      metric = m,
      n_packages = n(),
      pct_na_rows = pct_missing_observations, 
      share_single_value = round(mean(n_distinct == 1) * 100, 1),
      perc_variation = 100 - round(mean(n_distinct == 1) * 100, 1),
      n_fluctuating_packages = sum(n_distinct > 1),
      only_eq_10 = round(mean(all_ten) * 100, 1),
      only_eq_0 = round(mean(all_zero) * 100, 1)
    )
}) %>%
  bind_rows() %>%
  arrange(desc(only_eq_10))
 
print(within_var_summary, width = Inf)

# within_var_summary <- lapply(metrics, function(m) {
#   panel %>%
#     filter(!is.na(.data[[m]])) %>%
#     group_by(package_name) %>%
#     summarise(
#       n_obs = n(),
#       n_distinct = n_distinct(.data[[m]]),
#       within_sd = sd(.data[[m]]),
#       all_zero = all(.data[[m]] == 0),
#       all_ten = all(.data[[m]] == 10),
#       .groups = "drop"
#     ) %>%
#     summarise(
#       metric = m,
#       n_packages = n(),
#       share_single_value = mean(n_distinct == 1) * 100,
#       n_fluctuating_packages = sum(n_distinct > 1),
#       only_eq_10 = mean(all_ten) * 100,
#       only_eq_0 = mean(all_zero) * 100,
#     )
# }) %>%
#   bind_rows() %>%
#   arrange(desc(only_eq_10))
 
# print(within_var_summary, width = Inf)



# A tibble: 12 × 8
   metric                    n_packages pct_na_rows share_single_value
   <chr>                          <int>       <dbl>              <dbl>
 1 Binary.Artifacts_score         15598         0                 98.1
 2 License_score                  15598         0                 99.7
 3 Dangerous.Workflow_score       15598         7.2               95.3
 4 Contrib_score                  15565         0.9               93.6
 5 DependUpTool_score             15586         0.4               93.8
 6 Maintained_score               15572         0.5               30.8
 7 Security.Policy_score          15598         0                 97.7
 8 Code.Review_score              15598         0.4               43.8
 9 Token.Permissions_score        15598         7.2               93.2
10 Fuzzing_score                  15569         1.6               99.7
11 Pinned.Dependencies_score      15598         7.2               84.1
12 Aggregate_score                15598         0         

In [ ]:
latex_df <- within_var_summary %>%
  rename(
    `Metric` = metric,
    `\\% w/ Variation` = perc_variation,
    `\\% -1/Missing` = pct_na_rows,
    `\\% = 0` = only_eq_0,
    `\\% = 10` = only_eq_10
  ) %>%
  select(`Metric`, `\\% w/ Variation`, `\\% -1/Missing`, `\\% = 0`, `\\% = 10`)

latex_table <- kable(
  latex_df, 
  format = "latex", 
  booktabs = TRUE, 
  escape = FALSE, 
  align = "lccccc",
  caption = "\\textsc{Within-Package Variation in Scorecard Metrics (Grouped by Package)}"
)

cat(latex_table)


Attaching package: ‘kableExtra’


The following object is masked from ‘package:dplyr’:

    group_rows




\begin{table}

\caption{\textsc{Within-Package Variation in Scorecard Metrics (Grouped by Package)}}
\centering
\begin{tabular}[t]{lcccc}
\toprule
Metric & \% w/ Variation & \% -1/Missing & \% = 0 & \% = 10\\
\midrule
Binary.Artifacts_score & 1.9 & 0.0 & 0.5 & 95.3\\
License_score & 0.3 & 0.0 & 5.8 & 83.7\\
Dangerous.Workflow_score & 4.7 & 7.2 & 1.0 & 82.7\\
Contrib_score & 6.4 & 0.9 & 10.1 & 59.3\\
DependUpTool_score & 6.2 & 0.4 & 49.8 & 44.1\\
\addlinespace
Maintained_score & 69.2 & 0.5 & 0.3 & 29.4\\
Security.Policy_score & 2.3 & 0.0 & 82.2 & 12.4\\
Code.Review_score & 56.2 & 0.4 & 33.1 & 3.8\\
Token.Permissions_score & 6.8 & 7.2 & 78.1 & 2.2\\
Fuzzing_score & 0.3 & 1.6 & 98.8 & 0.9\\
\addlinespace
Pinned.Dependencies_score & 15.9 & 7.2 & 59.6 & 0.6\\
Aggregate_score & 94.0 & 0.0 & 0.0 & 0.0\\
\bottomrule
\end{tabular}
\end{table}

In [9]:
print("Aggregate_score distribution in monthly aggregated panel:")
summary(panel$Aggregate_score)
print("Aggregate_score distribution in raw data:")
summary(raw_df$Aggregate_score)

print("Maintained distribution in monthly aggregated panel:")
summary(panel$Maintained_score)
print("Maintained distribution in raw data:")
summary(raw_df$Maintained_score)

print("Code-review distribution in monthly aggregated panel:")
summary(panel$Code_Review_score)
print("Coder-review distribution in raw data:")
summary(raw_df$Code.Review_score)

range_dist <- lapply(metrics, function(m) {
  panel %>%
    filter(!is.na(.data[[m]])) %>%
    group_by(package_name) %>%
    summarise(range = max(.data[[m]]) - min(.data[[m]]), .groups = "drop") %>%
    summarise(metric        = m,
              n_changers = sum(range > 0),
              median_fluctuation     = median(range[range > 0]),
              p75_fluctuation        = quantile(range[range > 0], 0.75),
              p90_fluctuation        = quantile(range[range > 0], 0.90),
              share_fluctuation_ge_2 = round(100 * mean(range[range > 0] >= 2), 1))
}) %>% bind_rows() %>% arrange(median_fluctuation)

print(as_tibble(range_dist), n = Inf)


[1] "Aggregate_score distribution in monthly aggregated panel:"


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.700   4.400   5.300   5.275   6.100  10.000 

[1] "Aggregate_score distribution in raw data:"


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.700   4.600   5.500   5.504   6.300  10.000 

[1] "Maintained distribution in monthly aggregated panel:"


...

[1] "Code-review distribution in monthly aggregated panel:"


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
  0.000   0.000   2.000   3.296   6.000  10.000     210 

[1] "Coder-review distribution in raw data:"


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 -1.000   0.000   2.000   3.539   7.000  10.000 

# A tibble: 12 × 6
   metric          n_changers median_fluctuation p75_fluctuation p90_fluctuation
   <chr>                <int>              <dbl>           <dbl>           <dbl>
 1 Aggregate_score      13707              0.700               1            1.45
 2 Binary_Artifac…        297              1                   2            6   
 3 Code_Review_sc…       8429              2                   4            6   
 4 Pinned_Depende…       1813              2                   4            7   
 5 Contrib_score          628              4                   6            7   
 6 Maintained_sco…       9910              5                   7            9   
 7 License_score           46              9                   9            9   
 8 Token_Permissi…        553              9                  10           10   
 9 Dangerous_Work…        218             10                  10           10   
10 DependUpTool_s…        868             10                  10           10   
11 Fuzzin

For each metric, we compute the range, max minus min of that package's monthly score across all its observed months — and then summarize the distribution of those package-level ranges, restricted to packages where the range greater than 0. The table highlights "among packages whose score varies at all in the monthly panel, how far does it travel in total?"

The Aggregate score has negligible mass at the scores of 0 and 10, with 0 packages having a score of 0 or a score of 10 for all releases. The Maintained and Code-Review checks have less than 30% and 35% combined mass at the scores of 0 and 10, respectively. Thus, we model the Aggregate, Maintained, and Code-Review scores continuously and utilize binned specifications for other check-level metrics, where boundaries are indicitive of meaningful states.

In [19]:
skewness(panel$loc, na.rm = TRUE)
skewness(panel$dependents, na.rm = TRUE)
skewness(panel$downloads_7_day_total, na.rm = TRUE)
skewness(panel$version_age_days, na.rm = TRUE)
skewness(panel$Release_count)

## log (x + 1) to handle the skewness of LOC, dependents, download count, and release count
skewness(log1p(panel$loc), na.rm = TRUE)
skewness(log1p(panel$dependents), na.rm = TRUE)
skewness(log1p(panel$downloads_7_day_total), na.rm = TRUE)
skewness(log1p(panel$Release_count))


[1] 11.56025

[1] 190.5892

[1] 97.28395

[1] 0.7075794

[1] 74.9252

[1] 0.1898272

[1] 2.941587

[1] 0.9085436

[1] 2.030001